# วิเคราะห์ artifact รอบ pedicle screw ด้วย TONIQ

Notebook นี้พาวิเคราะห์ภาพ MRI ของ phantom ทีละขั้น ได้ผล 4 อย่างตามงานวิจัย TONIQ:

| ผลลัพธ์ | ความหมาย | ภาพที่ใช้ |
|---|---|---|
| **IA** (Intensity Artifact) | สัญญาณหาย (มืด) หรือสัญญาณกองรวม (สว่างเกิน) รอบสกรู | uniform plastic / metal |
| **GD** (Geometric Distortion) | ภาพเลื่อนตำแหน่งไปกี่ mm | structured plastic / metal |
| **SNR** | สัญญาณเทียบกับสัญญาณรบกวน | uniform metal สแกน 2 ครั้ง |
| **SR** (Spatial Resolution) | ภาพเบลอแค่ไหน (FWHM, mm) ไม่บังคับ ใช้เวลานาน | structured metal + reference ความละเอียดสูง |

## วิธีใช้
1. คลิกที่ช่อง (cell) ที่เป็นโค้ด แล้วกด **Shift + Enter** เพื่อรันและเลื่อนไปช่องถัดไป
2. รันเรียงจากบนลงล่าง **ห้ามข้ามขั้น**
3. ส่วนที่ต้องแก้มีแค่ช่องที่เขียนว่า **✏️ แก้ตรงนี้** เท่านั้น
4. ถ้าแก้ค่าแล้ว ให้รันช่องนั้นและช่องต่อจากนั้นใหม่

## ภาพที่ต้องสแกน (ทุก series ต้องใช้ FOV, matrix และตำแหน่งเดียวกัน)

| ชื่อในโปรแกรม | phantom | สกรู |
|---|---|---|
| `uniform-plastic` | uniform | พลาสติก (พิมพ์ 3D) |
| `uniform-plastic-2` | uniform | พลาสติก (สแกนซ้ำ ไม่บังคับ) |
| `uniform-metal` | uniform | โลหะจริง |
| `uniform-metal-2` | uniform | โลหะจริง (สแกนซ้ำ ใช้คำนวณ SNR) |
| `structured-plastic` | structured (lattice) | พลาสติก |
| `structured-metal` | structured (lattice) | โลหะจริง |
| `structured-plastic-reference` | structured | พลาสติก ความละเอียดสูง (ใช้คำนวณ SR) |

## ขั้นที่ 0: บอกโปรแกรมว่าไฟล์อยู่ที่ไหน ✏️ แก้ตรงนี้

- `EXPORT_FOLDER` คือโฟลเดอร์ที่ export ภาพ DICOM ออกมาจากเครื่อง MRI (โปรแกรมจะค้นในโฟลเดอร์ย่อยให้เอง)
- `SORTED_FOLDER` คือโฟลเดอร์ที่จะให้โปรแกรมจัดไฟล์แยกตาม series (สร้างให้อัตโนมัติ)
- `RESULTS_FOLDER` คือโฟลเดอร์เก็บรูปและตารางผลลัพธ์

วิธีคัดลอก path บน Windows: เปิดโฟลเดอร์ใน File Explorer แล้วคลิกที่แถบที่อยู่ด้านบน จะได้ข้อความเช่น `C:\TONIQ\export` คัดลอกมาวางไว้ระหว่าง `r'` กับ `'`

In [ ]:
EXPORT_FOLDER  = r'C:\TONIQ\export'
SORTED_FOLDER  = r'C:\TONIQ\sorted'
RESULTS_FOLDER = r'C:\TONIQ\results'

In [ ]:
# เตรียมเครื่องมือ (ไม่ต้องแก้)
import sys, os
sys.path.insert(0, os.path.abspath('..'))
%matplotlib inline
import toniq_easy as te
print('พร้อมใช้งาน')

## ขั้นที่ 1: จัดไฟล์แยกตาม series

โปรแกรมจะคัดลอกภาพ MR จาก `EXPORT_FOLDER` ไปไว้ในโฟลเดอร์ชื่อ `Series<เลข series>` แล้วแสดงตารางรายชื่อ series ให้ดู
ใช้ตารางนี้กรอกเลข series ในขั้นที่ 2 (รันซ้ำได้ ไม่เสียหาย)

In [ ]:
series_table = te.sort_export(EXPORT_FOLDER, SORTED_FOLDER)

## ขั้นที่ 2: บอกว่า series ไหนคือภาพอะไร ✏️ แก้ตรงนี้

ดูชื่อ protocol จากตารางในขั้นที่ 1 แล้วใส่ชื่อโฟลเดอร์ให้ตรงกับภาพแต่ละแบบ
ถ้าไม่ได้สแกนภาพไหน ให้ใส่ `None` (ไม่ต้องมีเครื่องหมายคำพูด) ขั้นที่ต้องใช้ภาพนั้นจะถูกข้ามไป

In [ ]:
SERIES = {
    'uniform-plastic':              'Series301',
    'uniform-metal':                'Series501',
    'uniform-metal-2':              'Series601',
    'structured-plastic':           'Series701',
    'structured-metal':             'Series901',
    'structured-plastic-reference': None,
}
volumes = te.load_volumes(SORTED_FOLDER, SERIES)
resolution_mm = volumes['uniform-metal'].meta.resolution_mm

## ขั้นที่ 3: เลือกขอบเขตที่จะวิเคราะห์ (crop) ✏️ แก้ตรงนี้

รอบแรกให้รันโดยใช้ `CROP = None` ก่อน แล้วดูภาพ 3 มุมมอง จากนั้นอ่านตัวเลขบนแกนเพื่อกำหนดกรอบใหม่
รูปแบบคือ `[[เริ่ม, จบ] ของ axis 0, [เริ่ม, จบ] ของ axis 1, [เริ่ม, จบ] ของ axis 2]`
แล้วรันช่องนี้ซ้ำจนกรอบสีเหลืองครอบบริเวณที่ต้องการ

**หลักการเลือกกรอบ**
- ครอบตัวสกรูและบริเวณรอบๆ ที่มี artifact ทั้งหมด โดยให้สกรูอยู่กลางกรอบ
- **ไม่**ครอบผนังกล่อง โครงยึด หรืออากาศ
- ไม่ควรเล็กกว่าประมาณ 40 × 40 × 20 ช่องภาพ เพราะการคำนวณ SNR และ SR ใช้หน้าต่างขนาดราว 10–14 ช่องภาพ

In [ ]:
CROP = None
# ตัวอย่าง: CROP = [[30, 163], [61, 194], [14, 48]]

te.show_overview(volumes, CROP, save_folder=RESULTS_FOLDER)

## ขั้นที่ 4: Intensity Artifact (IA)

สีแดงคือสัญญาณกองรวม (สว่างเกิน) สีน้ำเงินคือสัญญาณหาย ส่วนพื้นที่ลายขาวคือตัวสกรูซึ่งไม่ได้นำมาคำนวณ
`IA_THRESHOLD = 0.3` หมายถึงนับว่าเป็น artifact เมื่อสัญญาณเปลี่ยนเกิน 30% ตามแนวทางของ ASTM F2119

In [ ]:
IA_THRESHOLD = 0.3

ia_result = te.run_ia(volumes, CROP)
te.show_ia(ia_result, save_folder=RESULTS_FOLDER)

**อยากดู slice อื่น:** เปลี่ยน `index` (ลำดับ slice) และ `axis` (0, 1 หรือ 2 คือทิศของภาพตัดขวาง) แล้วรันช่องนี้ซ้ำได้เรื่อยๆ
ภาพตัดตามแนวยาวของสกรูมักเห็น artifact ที่ปลายสกรูได้ชัดที่สุด

In [ ]:
te.show_ia(ia_result, index=None, axis=2, save_folder=RESULTS_FOLDER)

## ขั้นที่ 5: Geometric Distortion (GD)

โปรแกรมนำภาพ structured plastic ไปบิดให้ซ้อนกับภาพ structured metal แล้ววัดว่าแต่ละจุดเลื่อนไปกี่ช่องภาพ
ขั้นนี้ใช้เวลาหลายนาที ทิศที่มักเลื่อนมากที่สุดคือทิศ readout (frequency encoding)

In [ ]:
gd_result = None
if SERIES.get('structured-plastic') and SERIES.get('structured-metal'):
    gd_result = te.run_gd(volumes, CROP, ia_result, IA_THRESHOLD)
    te.show_gd(gd_result, save_folder=RESULTS_FOLDER)
else:
    print('ข้ามขั้นนี้: ไม่มีภาพ structured')

## ขั้นที่ 6: SNR

ใช้ภาพ uniform metal ที่สแกนซ้ำ 2 ครั้ง บริเวณที่มี artifact มากจะถูกตัดออกจากการคำนวณ (แสดงเป็นลายขาว)
ถ้าสีในภาพอิ่มตัวทั้งภาพ ให้ปรับ `ticks` ให้เหมาะกับค่า SNR ของเครื่อง 1.5T ของคุณ

In [ ]:
snr_result = None
if SERIES.get('uniform-metal-2'):
    snr_result = te.run_snr(volumes, CROP, ia_result, IA_THRESHOLD)
    te.show_snr(snr_result, ticks=(0, 50, 100), save_folder=RESULTS_FOLDER)
else:
    print('ข้ามขั้นนี้: ไม่มีภาพ uniform-metal-2')

## ขั้นที่ 7: Spatial Resolution (SR) ไม่บังคับ ✏️

ขั้นนี้ช้ามาก อาจใช้เวลาหลายสิบนาทีถึงหลายชั่วโมง ขึ้นกับขนาดกรอบ crop และความเร็วคอมพิวเตอร์
ถ้าต้องการรัน ให้เปลี่ยน `RUN_SR = False` เป็น `RUN_SR = True` ต้องมีภาพ `structured-plastic-reference` และต้องรันขั้นที่ 5 ผ่านแล้ว

บริเวณที่เป็นสีขาวว่างคือส่วนที่วัดไม่ได้ (ใกล้สกรูหรือมี artifact มาก) โปรแกรมจะเลือกแสดง slice ที่วัดได้มากที่สุดให้เอง

In [ ]:
RUN_SR = False

sr_result = None
if RUN_SR and gd_result is not None and SERIES.get('structured-plastic-reference'):
    sr_result = te.run_sr(volumes, CROP, ia_result, gd_result, IA_THRESHOLD)
    te.show_sr(sr_result, resolution_mm, save_folder=RESULTS_FOLDER)
else:
    print('ข้ามขั้นนี้')

## ขั้นที่ 8: สรุปตัวเลขและบันทึกผล

ได้ตาราง `summary.csv` (เปิดด้วย Excel ได้) รูปทุกรูปเป็นไฟล์ `.png` และข้อมูล map เป็นไฟล์ `.npy` ทั้งหมดอยู่ใน `RESULTS_FOLDER`
ตัวเลขเหล่านี้ใช้เปรียบเทียบกันได้ เช่น ระหว่างสกรูต่างวัสดุ ต่างแนวการวางเทียบกับสนามแม่เหล็ก หรือต่าง protocol

In [ ]:
summary = te.summarize(resolution_mm, ia_result, IA_THRESHOLD, gd_result, snr_result, sr_result, save_folder=RESULTS_FOLDER)
te.save_maps(RESULTS_FOLDER, ia=ia_result, gd=gd_result, snr=snr_result, sr=sr_result)

## เมื่อเจอปัญหา

| ข้อความ | สาเหตุและวิธีแก้ |
|---|---|
| `ไม่พบไฟล์ภาพ MR` | path ใน `EXPORT_FOLDER` ผิด ให้คัดลอกจาก File Explorer ใหม่ |
| `FileNotFoundError ... SeriesXXX` | ชื่อโฟลเดอร์ใน `SERIES` ไม่ตรงกับตารางในขั้นที่ 1 |
| `ภาพแต่ละ series มีขนาดไม่เท่ากัน` | สแกนด้วย FOV หรือ matrix ไม่เท่ากัน ต้องสแกนใหม่โดยคัดลอก prescription เดียวกัน |
| `Multiple images at the same slice position` | series มีทั้งภาพ magnitude และ phase ให้ export เฉพาะ magnitude |
| `NameError: name 'ia_result' is not defined` | ข้ามขั้นไป ให้รันขั้นก่อนหน้าให้ครบ |
| IA ทั้งภาพติดลบหรือบวกเท่าๆ กัน | phantom ขยับ หรือตั้งค่าการสแกนไม่เหมือนกันระหว่างภาพ plastic กับ metal |